In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_msd_1 = pd.read_csv('msd-publication/data_1.csv')
df_msd_2 = pd.read_csv('msd-publication/data_2.csv')
df_msd = pd.concat([df_msd_1, df_msd_2], axis=0)
df_msd['SMILES'] = df_msd.smiles
df_msd = df_msd.drop(columns=['smiles'])
del df_msd_1, df_msd_2
df_galapagos = pd.read_csv('galapagos-publication/data.csv')

In [ ]:
from rdkit import Chem
from scipy.stats import skew

# ── 1. Canonicalize SMILES ────────────────────────────────────────────────────

def canonicalize_smiles(smi):
    """Return canonical SMILES or NaN if unparseable."""
    if pd.isna(smi):
        return np.nan
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return np.nan
    return Chem.MolToSmiles(mol)

for name, df in [("df_msd", df_msd), ("df_galapagos", df_galapagos)]:
    df["SMILES"] = df["SMILES"].apply(canonicalize_smiles)
    n_bad = df["SMILES"].isna().sum()
    print(f"{name}: {len(df)} rows, {n_bad} unparseable SMILES dropped")

# drop rows with unparseable SMILES
df_msd       = df_msd.dropna(subset=["SMILES"]).reset_index(drop=True)
df_galapagos = df_galapagos.dropna(subset=["SMILES"]).reset_index(drop=True)

print("\n✅ SMILES canonicalized")

In [ ]:
# ── 2. Drop all-string columns (except SMILES) & convert numeric strings ──────

def clean_columns(df, df_name):
    """
    For every non-SMILES column:
      - If ALL non-NaN values are strings that can't be cast to float → drop it.
      - Otherwise try to cast to float (handles '5', '3.321', etc.).
    """
    cols_to_drop = []
    cols_converted = []

    for col in df.columns:
        if col == "SMILES":
            continue

        non_null = df[col].dropna()
        if len(non_null) == 0:
            cols_to_drop.append(col)
            continue

        # try converting the whole column to numeric
        converted = pd.to_numeric(non_null, errors="coerce")

        # if ALL non-null values become NaN after conversion → pure string column
        if converted.isna().all():
            cols_to_drop.append(col)
        else:
            # some or all values converted; cast column
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if df[col].dtype != np.float64:
                df[col] = df[col].astype(float)
            cols_converted.append(col)

    if cols_to_drop:
        print(f"  {df_name}: dropping string columns → {cols_to_drop}")
    df.drop(columns=cols_to_drop, inplace=True)
    return df

for name, df in [("df_msd", df_msd), ("df_galapagos", df_galapagos)]:
    clean_columns(df, name)

print("\n✅ String columns dropped, remaining columns cast to float")

In [ ]:
# ── 3. Skew correction: log10(y + 1) for highly skewed positive / negative cols ─

SKEW_THRESHOLD = 1.2  # |skewness| above this triggers transform

def skew_correct(df, df_name):
    """
    For each numeric column (excl. SMILES):
      - Compute skewness on non-NaN values.
      - If always ≥ 0 and skew > threshold  →  log10(y + 1)
      - If always ≤ 0 and skew < -threshold →  flip sign, then log10(y + 1)
    """
    transformed = []
    for col in df.columns:
        if col == "SMILES":
            continue
        vals = df[col].dropna()
        if len(vals) < 10:
            continue

        sk = skew(vals, nan_policy="omit")
        vmin, vmax = vals.min(), vals.max()

        if vmin >= 0 and sk > SKEW_THRESHOLD:
            df[col] = np.log10(df[col] + 1)
            transformed.append((col, f"skew={sk:.2f}, all≥0 → log10(y+1)"))
        elif vmax <= 0 and sk < -SKEW_THRESHOLD:
            df[col] = np.log10(-df[col] + 1)
            transformed.append((col, f"skew={sk:.2f}, all≤0 → flip + log10(y+1)"))

    if transformed:
        print(f"\n  {df_name}: transformed {len(transformed)} columns:")
        for c, reason in transformed:
            print(f"    {c} ({reason})")
    else:
        print(f"\n  {df_name}: no columns required skew correction")
    return df

for name_ref in [("df_msd", df_msd), ("df_galapagos", df_galapagos)]:
    skew_correct(name_ref[1], name_ref[0])

print("\n✅ Skew correction done")

In [ ]:
# ── 4. Check for duplicate columns between df_msd and df_galapagos ────────────
#    Keep redundant columns only in df_msd (drop from df_galapagos)

msd_cols = set(df_msd.columns) - {"SMILES"}
gal_cols = set(df_galapagos.columns) - {"SMILES"}

# Find columns that overlap by exact name
exact_overlap = msd_cols & gal_cols
print(f"Exact name overlap: {exact_overlap if exact_overlap else 'none'}")

# Also check semantic overlap: merge on SMILES and compute correlations
# between similarly-named columns to find true duplicates
shared_smiles = set(df_msd["SMILES"]) & set(df_galapagos["SMILES"])
print(f"\nShared molecules between MSD & Galapagos: {len(shared_smiles)}")

if len(shared_smiles) > 100:
    merged_check = pd.merge(
        df_msd, df_galapagos, on="SMILES", how="inner", suffixes=("_msd", "_gal")
    )
    
    # Find highly correlated column pairs (potential duplicates)
    dup_pairs = []
    for mc in msd_cols:
        for gc in gal_cols:
            if mc == gc:
                continue
            col_m = f"{mc}_msd" if mc in exact_overlap else mc
            col_g = f"{gc}_gal" if gc in exact_overlap else gc
            if col_m not in merged_check.columns or col_g not in merged_check.columns:
                continue
            valid = merged_check[[col_m, col_g]].dropna()
            if len(valid) < 50:
                continue
            corr = valid[col_m].corr(valid[col_g])
            if abs(corr) > 0.85:
                dup_pairs.append((mc, gc, corr, len(valid)))
    
    if dup_pairs:
        print(f"\nHighly correlated cross-dataset column pairs (|r| > 0.85):")
        for mc, gc, corr, n in sorted(dup_pairs, key=lambda x: -abs(x[2])):
            print(f"  MSD: {mc}  <->  GAL: {gc}  (r={corr:.3f}, n={n})")
        
        # Drop the Galapagos version of correlated pairs
        gal_to_drop = list(set(gc for _, gc, _, _ in dup_pairs))
        print(f"\nDropping from df_galapagos (keeping MSD version): {gal_to_drop}")
        df_galapagos = df_galapagos.drop(columns=gal_to_drop)
    else:
        print("\nNo highly correlated duplicate columns found.")

# Also drop exact-name overlaps from galapagos
if exact_overlap:
    print(f"\nDropping exact-name overlaps from df_galapagos: {exact_overlap}")
    df_galapagos = df_galapagos.drop(columns=list(exact_overlap))

print("\n✅ Duplicate column check done")

In [ ]:
# ── 5. Aggregate per SMILES (mean) and merge on canonical SMILES ──────────────

# Within each dataframe, aggregate duplicate SMILES by taking the mean
dfs = {
    "msd": df_msd,
    "galapagos": df_galapagos,
}

agg_dfs = {}
for name, df in dfs.items():
    n_before = len(df)
    df_agg = df.groupby("SMILES", as_index=False).mean(numeric_only=True)
    n_after = len(df_agg)
    print(f"{name}: {n_before} → {n_after} unique SMILES ({df_agg.shape[1]-1} endpoint cols)")
    agg_dfs[name] = df_agg

# Outer merge all dataframes on SMILES
df_final = agg_dfs["msd"]
for name in ["galapagos"]:
    df_final = pd.merge(df_final, agg_dfs[name], on="SMILES", how="outer")

print(f"\n📊 Final merged dataframe: {df_final.shape[0]} molecules × {df_final.shape[1]-1} ADME endpoints")
print(f"   SMILES column + {df_final.shape[1]-1} float endpoint columns")
print(f"   Sparsity: {df_final.drop(columns='SMILES').isna().mean().mean()*100:.1f}% NaN")

df_final

In [ ]:
# ── 5b. Add "_endpoint" suffix to all non-SMILES columns ──────────────────────

rename_map = {
    col: col if col.endswith("_endpoint") else f"{col}_endpoint"
    for col in df_final.columns if col != "SMILES"
}
df_final = df_final.rename(columns=rename_map)

n_renamed = sum(1 for old, new in rename_map.items() if old != new)
print(f"Renamed {n_renamed} columns (added '_endpoint' suffix)")
print(f"Columns already with suffix kept as-is: {sum(1 for old, new in rename_map.items() if old == new)}")
print(f"\n✅ All {len(rename_map)} endpoint columns now have '_endpoint' suffix")

In [ ]:
# ── 5c. Drop molecules with MW > 1400 or fewer than 4 heavy atoms ─────────────

from rdkit.Chem import Descriptors

MW_MAX = 1400
MIN_HEAVY_ATOMS = 4

n_before = len(df_final)

mw_list = []
ha_list = []
keep_mask = []

for smi in df_final["SMILES"]:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        mw_list.append(np.nan)
        ha_list.append(np.nan)
        keep_mask.append(False)
    else:
        mw = Descriptors.MolWt(mol)
        ha = mol.GetNumHeavyAtoms()
        mw_list.append(mw)
        ha_list.append(ha)
        keep_mask.append(mw <= MW_MAX and ha >= MIN_HEAVY_ATOMS)

df_final["_mw"] = mw_list
df_final["_ha"] = ha_list

n_high_mw = (df_final["_mw"] > MW_MAX).sum()
n_low_ha = (df_final["_ha"] < MIN_HEAVY_ATOMS).sum()

df_final = df_final[keep_mask].reset_index(drop=True)
df_final = df_final.drop(columns=["_mw", "_ha"])

n_after = len(df_final)
print(f"Dropped {n_high_mw} molecules with MW > {MW_MAX}")
print(f"Dropped {n_low_ha} molecules with < {MIN_HEAVY_ATOMS} heavy atoms")
print(f"Total removed: {n_before - n_after} molecules ({n_before} → {n_after})")
print(f"\n✅ Molecular weight & heavy atom filter done")

In [ ]:
# ── 6. Final validation and save data.csv ──────────────────────────────────────

# Verify all endpoint columns are float
non_float = [c for c in df_final.columns if c != "SMILES" and df_final[c].dtype != np.float64]
assert len(non_float) == 0, f"Non-float columns remain: {non_float}"

# ── Save ──────────────────────────────────────────────────────────────────────
df_final.to_csv("data.csv", index=False)
print(f"\n💾 Saved data.csv  ({df_final.shape[0]} rows × {df_final.shape[1]} cols)")

# ── Print summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Molecules:  {df_final.shape[0]:,}")
print(f"  Endpoints:  {df_final.shape[1] - 1}")
print(f"  Sparsity:   {round(float(df_final.drop(columns='SMILES').isna().mean().mean() * 100), 1)}% NaN")
print(f"{'='*70}")
print("\n✅ Done")